# Purpose: make a space for discriminating learners from non, controls and mutants

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq 
from mapd.sinq_builders import build_composite_sinq, subset_sinq
import h5py
import glob
import numpy as np
import pickle
import plotly.express as px
import matplotlib.colors as mcolors
# import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import seaborn as sns

import random

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

%load_ext autoreload
%autoreload 2

import mapd.sentinels  # load once
%aimport -mapd.sentinels

%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)


# Fig dir for PCA analysis
pca_fig_dir = 'Figure3_6_PCA_LDA'
control_dir = f'{pca_fig_dir}/controls'
mutants_dir = f'{pca_fig_dir}/mutants'

control_export_dir = f"{control_dir}/exports"
mutants_export_dir = f"{mutants_dir}/exports"

factor_plots_dir = f'{pca_fig_dir}/factor_plots'

os.makedirs(pca_fig_dir, exist_ok=True)
os.makedirs(control_dir, exist_ok=True)
os.makedirs(mutants_dir, exist_ok=True)
os.makedirs(control_export_dir, exist_ok=True)
os.makedirs(mutants_export_dir, exist_ok=True)
os.makedirs(f'{control_dir}/expts_in_LDA_space', exist_ok=True)
os.makedirs(factor_plots_dir, exist_ok=True)

# def mk_dir(id):
#     folder_path = f'./{fig_dir}/{id}'
#     os.makedirs(folder_path, exist_ok=True)
#     export_path = f'{folder_path}/exports'
#     os.makedirs(export_path, exist_ok=True)

# control_dir


# Load learners vs. control sinq

In [ ]:
sinq_0 = Sinq(sinqname='Figure1')
sinq_1 = Sinq(sinqname='Fig1_control')
sinq = build_composite_sinq(name='Figure2_learn_v_no_learn_lda',sources=[sinq_0,sinq_1])

In [ ]:
op_df = sinq.display_df(show_tags=True)
op_df['learn'] = sinq.has_tag('learn')
op_df['control'] = sinq.has_tag('control')
op_df[['genotype','tags','learn','control']]

# Compare learners vs. not

In [ ]:
# print([i for i in op_df['genotype'].unique()])
list(op_df['genotype'].unique())

In [ ]:
geno_map = {'Hot-Cell-Gal4 (test)':             'HC',
 'Hot-Cell-LexA_Chr;81A06_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;35c09_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;35C09_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;78E05_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;31H05_pJFRC7': 'HC',
 'SS61350_pJFRC7':                  '+',
 '+;31H05-Gal4_pJFRC7':             '+',
 '+;TH-Gal4_UAS-Kir2_1':            '+;TH-Gal4_UAS-Kir2_1',
 '+;pJFRC7;31H05-Gal4':             '+',
 '+;pJFRC7;SS61350':                '+',
 'ppk-Gal4;10XUAS-ChR in WT':       '+;ppk-GAL4;10XUAS-ChR',
 '+;ppk-GAL4;10XUAS-ChR':           '+;ppk-GAL4;10XUAS-ChR',
 'Hot-Cell-Gal4_ChrinWT':           'HC',
 'w, NorpA':                        'NorpA',
 '+;pJFRC7;+':                      '+',
 'norpA;cry':                       '+',
 '+;pJFRC7;35C09-GAL4':             '+',
 'ss61350_pJFRC7':                  '+',
 'norpAE55':                        'NorpA',
 '+;pFJRC7;+':                      '+',
 'norpA':                           'NorpA',
 '+;pJFRC7;81A06-GAL4':             '+',
}

op_df['geno_smpl'] = op_df['genotype'].map(geno_map)
op_df.loc[op_df['geno_smpl'].isna()]

## Additional factors

In [ ]:
# op_df.loc['250506_F2_C1','learn'] = True
# op_df.loc['250506_F2_C1','learn_note'] = 'ok learner, got frustrated a bit'

# op_df.loc['250514_F1_C1','learn'] = True
# op_df.loc['250514_F1_C1','learn_note'] = 'ok learner, timed out a bit'

# op_df.loc['251219_F1_C2','learn'] = True
# op_df.loc['251219_F1_C2','learn_note'] = 'easy targets'

# # Don't learn for some reason

# op_df.loc['250304_F2_C1','learn'] = False
# op_df.loc['250304_F2_C1','learn_note'] = 'green, frustrated'

# op_df.loc['210331_F2_C1','learn'] = False
# op_df.loc['210331_F2_C1','learn_note'] = 'Not sure...'

# op_df.loc['250226_F2_C1','learn'] = False
# op_df.loc['250226_F2_C1','learn_note'] = 'Never got it, tried hard'

# op_df.loc['250228_F1_C1','learn'] = False
# op_df.loc['250228_F1_C1','learn_note'] = 'Adjusted baseline, but never got it'

# op_df.loc['250923_F1_C1','learn'] = False
# op_df.loc['250923_F1_C1','learn_note'] = 'learns late?'

# op_df.loc['250924_F1_C1','learn'] = False
# op_df.loc['250924_F1_C1','learn_note'] = 'tires out'

# op_df.loc['250925_F2_C1','learn'] = False
# op_df.loc['250925_F2_C1','learn_note'] = 'tires out'

op_df.loc[op_df['learn'], 'learn_class'] = 0  # Control
op_df.loc[op_df['control'], 'learn_class'] = 1  # Control
op_df.loc[(op_df['control'] == op_df['learn']), 'learn_class'] = 2  # borderline cases

In [ ]:
op_df['success_rate'] =         op_df['successes'] / op_df['num_trials']
op_df['hard_success_rate'] =    op_df['hard_successes'] / op_df['num_trials']

op_df['as_off_over_successes'] =        np.log10(op_df['outcome_fractions_as_off'] * op_df['num_trials'] + 1) / (op_df['successes'] + 1)
op_df['as_off_over_hard_successes'] =   np.log10(op_df['outcome_fractions_as_off'] * op_df['num_trials'] + 1) / (op_df['hard_successes'] + 1)

op_df['rms_velocity_bar'] = op_df['rms_velocity'] / op_df['duration']
op_df['holding_cost_bar'] = op_df['holding_cost'] / op_df['duration']
op_df['probe_positive_effort_bar'] = op_df['probe_positive_effort'] / op_df['duration']

In [ ]:
op_df

# PCA on all factors

In [ ]:

# import matplotlib.pyplot as plt

# folder_path = "./Figure2/exports"
# figure_path = "./Figure2/"

# labels = ['learn_class','learn','note_text','geno_smpl']

# factors = [
#     'outcome_fractions_no_as_no_mv',
#     'outcome_fractions_no_as_mv',  
#     'outcome_fractions_as_off',
#     'outcome_fractions_as_off_late',
#     'outcome_fractions_timeout',
#     'hi_lo_shift',
#     'lo_state_on_target',
#     'hi_state_on_target',
#     'lo_target_off_state',
#     'hi_target_off_state',
#     'success_rate',
#     'hard_success_rate',
#     'as_off_over_successes',
#     'as_off_over_hard_successes',
#     'rms_velocity_bar',
#     'holding_cost_bar',
#     'probe_positive_effort_bar',
# ]
# pca_df = op_df.loc[:,labels+factors]

# # y = op_df['learners']
# X = pca_df.drop(columns=labels)

# scaler = StandardScaler()
# x_scaled = scaler.fit_transform(X,)

# pca = PCA(n_components=10,
#           copy=True, 
#           whiten=False, 
#           svd_solver='full', 
#           tol=0.0, 
#           iterated_power='auto')
# X_pca = pca.fit_transform(x_scaled)


# # Variance ratios
# vr = pca.explained_variance_ratio_
# cum = np.cumsum(vr)

# # How much variance in top 3?
# print(f"Top 4 PCs capture: {cum[3]:.2%} of variance")

# # Make a projection DataFrame
# proj_df = pd.DataFrame(
#     X_pca,
#     columns=[f"PC{i+1}" for i in range(pca.n_components_)],
#     index=pca_df.index
# )

# proj_df[labels] = pca_df[labels]
# col_str = 'learn_class'

# cats = pd.Index(sorted(proj_df[col_str].unique()))
# n = len(cats)
# palette = sns.color_palette("tab10", n)
# color_map = dict(zip(cats, palette))

# # Reuse proj_df and color_map from above
# plot_df = proj_df.reset_index().rename(columns={"index":"fly_id"})

# # Convert seaborn/matplotlib colors to hex for plotly
# color_map = dict(zip(cats, palette))
# color_map_hex = {g: mcolors.to_hex(c) for g, c in color_map.items()}

# fig = px.scatter_3d(
#     plot_df,
#     x="PC1", y="PC2", z="PC3",
#     color=col_str,
#     color_discrete_map=color_map_hex,
#     hover_data=["fly_id", "geno_smpl",'note_text'],
#     title="PCA Projections (PC1–PC3)"
# )
# fig.update_traces(marker=dict(size=5, opacity=0.9))
# fig.update_layout(
#     legend_title_text="Genotype",
#     scene=dict(
#         xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
#         yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
#         zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})",
#     )
# )

# fig.update_layout(
#     width=1100, height=850,                    # << size here
#     margin=dict(l=60, r=260, t=70, b=50),      # leave space for a tall legend
#     font=dict(size=16),
#     legend=dict(y=1, x=1.02, yanchor="top"),   # legend outside
#     scene=dict(
#         xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
#         yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
#         zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})",
#         xaxis=dict(tickfont=dict(size=12)),
#         yaxis=dict(tickfont=dict(size=12)),
#         zaxis=dict(tickfont=dict(size=12)),
#     ),
#     hoverlabel=dict(font_size=14)
# )

# fig.show()
# for leg in [False,True]:
#     for col in ['learn','geno_smpl']:
#         cats = pd.Index(sorted(proj_df[col].unique()))
#         n = len(cats)
#         palette = sns.color_palette("tab20", n) if n <= 20 else sns.husl_palette(n)
#         color_map = dict(zip(cats, palette))
#         point_colors = [mcolors.to_hex(color_map[g]) for g in proj_df[col]]

#         # --- Helper to save a 2D scatter as SVG ---
#         def save_pc2d(proj, pca_obj, xpc="PC1", ypc="PC2", fname="plot.svg"):
#             fig, ax = plt.subplots(figsize=(7.5, 6))
#             ax.scatter(proj[xpc], proj[ypc], s=50, alpha=0.9, c=point_colors)

#             # Axis labels with variance explained
#             pc_names = ["PC1","PC2","PC3"]
#             exp = {pc_names[i]: pca_obj.explained_variance_ratio_[i] for i in range(3)}
#             ax.set_xlabel(f"{xpc} ({exp[xpc]:.1%} var)")
#             ax.set_ylabel(f"{ypc} ({exp[ypc]:.1%} var)")
#             ax.set_title(f"PCA: {ypc} vs {xpc}")

#             # Legend (handles many genotypes)
#             handles = [plt.Line2D([0],[0], marker='o', linestyle='', markersize=7,
#                                 markerfacecolor=mcolors.to_hex(color_map[g])) for g in cats]
#             ncol = 1 if n < 12 else 2 if n <= 24 else 3
#             if leg:
#                 ax.legend(handles, list(cats), title=col,
#                         bbox_to_anchor=(1.02, 1), loc="upper left", ncol=ncol, fontsize=8)

#             ax.grid(True, alpha=0.2)
#             plt.tight_layout()
#             fig.savefig(fname, format="svg", dpi=300, bbox_inches="tight")
#             plt.close(fig)
#             print(f"Saved {fname}")

#         # --- Make the two figures ---
#         save_pc2d(proj_df, pca, xpc="PC1", ypc="PC2", fname=f"./Figure2/pca_PC2_vs_PC1_{col}_legend_{leg}.svg")
#         save_pc2d(proj_df, pca, xpc="PC1", ypc="PC3", fname=f"./Figure2/pca_PC3_vs_PC1_{col}_legend_{leg}.svg")

# loadings = pd.DataFrame(
#     pca.components_,
#     index=[f"PC{i+1}" for i in range(pca.n_components_)],
#     columns=X.columns
# )

# # Optional: variance-scaled loadings (correlations) sometimes used for “importance”:
# # These are eigenvectors scaled by sqrt(eigenvalues)
# var_scaled_loadings = pd.DataFrame(
#     (pca.components_.T * np.sqrt(pca.explained_variance_)).T,
#     index=[f"PC{i+1}" for i in range(pca.n_components_)],
#     columns=X.columns
# )

# # --- 4) Plot PC1 loadings sorted by absolute magnitude ---
# for pc in ["PC1",'PC2','PC3']:
#     pc_l = loadings.loc[pc].sort_values(key=np.abs, ascending=False)

#     fig = plt.figure(figsize=(10, 5))
#     pc_l.plot(kind="bar")
#     plt.title(f"{pc} Loadings (feature weights)")
#     plt.ylabel("Loading")
#     plt.xlabel("Features (sorted by |loading|)")
#     plt.tight_layout()

#     # If you want the top-N features programmatically:
#     topN = 15
#     # pc1_top = pc_l.head(topN)
#     # print(pc1_top)
#     fig.savefig(f'./Figure2/{pc}.svg')

# PCA on a subset of factors

In [ ]:
labels = ['geno_smpl','learn_class','learn','control','note_text']

factors = [
    'outcome_fractions_no_as_no_mv',        # pc1, 3
    #'outcome_fractions_no_as_mv',          # pc3, 2
    'outcome_fractions_as_off',            # pc2, 1
    #'outcome_fractions_as_off_late',
    #'outcome_fractions_timeout',           # pc2, 4
    #'hi_lo_shift',
    'lo_state_on_target',                   # pc1, 2
    'hi_state_on_target',                   # pc1, 1
    'lo_target_off_state',                  # pc3, 1
    'hi_target_off_state',                  # pc3, 4
    'success_rate',                         # pc2, 5
    'hard_success_rate',                    # pc1, 4
    'as_off_over_successes',                # pc3, 4
    'as_off_over_hard_successes',           # pc1, 5
    'rms_velocity_bar',                     # pc2, 3
    #'holding_cost_bar',
    'probe_positive_effort_bar',            # pc2, 2, pc3, 5
]
pca_df = op_df.loc[:,labels+factors]

# y = op_df['learners']
X = pca_df.drop(columns=labels)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(X,)

pca = PCA(n_components=10,
          copy=True, 
          whiten=False, 
          svd_solver='full', 
          tol=0.0, 
          iterated_power='auto')
X_pca = pca.fit_transform(x_scaled)


# Variance ratios
vr = pca.explained_variance_ratio_
cum = np.cumsum(vr)

# How much variance in top 3?
print(f"Top 4 PCs capture: {cum[6]:.2%} of variance")

# Make a projection DataFrame
proj_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)],
    index=pca_df.index
)

proj_df[labels] = pca_df[labels]

In [ ]:
cum

In [ ]:
from IPython.display import display

fig = Figure(figsize=(4,4))
# plt.figure(figsize=(5,5))
ax = fig.add_subplot(1, 1, 1)
ax.scatter([i for i in range(pca.n_components_)],cum)
ax.set_ylim([0, 1.1])
ax.set_ylabel('Cum Var')
display(fig)

fig.savefig(f'{control_dir}/variance_by_PC.svg')

In [ ]:
import plotly.io as pio
from IPython.display import display


col_str = 'learn'

cats = pd.Index(sorted(proj_df[col_str].unique()))
n = len(cats)
palette = sns.color_palette("tab10", n)
color_map = dict(zip(cats, palette))

# Reuse proj_df and color_map from above
plot_df = proj_df.reset_index().rename(columns={"index":"fly_id"})

# Convert seaborn/matplotlib colors to hex for plotly
color_map = dict(zip(cats, palette))
color_map_hex = {g: mcolors.to_hex(c) for g, c in color_map.items()}

fig = px.scatter_3d(
    plot_df,
    x="PC1", y="PC2", z="PC3",
    color=col_str,
    color_discrete_map=color_map_hex,
    hover_data=["fly_id", "geno_smpl",'note_text'],
    title="PCA Projections (PC1–PC3)"
)
fig.update_traces(marker=dict(size=5, opacity=0.9))
fig.update_layout(
    legend_title_text=col_str,
    scene=dict(
        xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})",
    )
)

fig.update_layout(
    width=1100, height=850,                    # << size here
    margin=dict(l=60, r=260, t=70, b=50),      # leave space for a tall legend
    font=dict(family="Arial",size=16),
    legend=dict(y=1, x=1.02, yanchor="top"),   # legend outside
    scene=dict(
        xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})",
        xaxis=dict(tickfont=dict(size=12)),
        yaxis=dict(tickfont=dict(size=12)),
        zaxis=dict(tickfont=dict(size=12)),
    ),
    hoverlabel=dict(font_size=14)
)


fig.update_layout(
    scene_camera=dict(
        eye=dict(x=-1.6, y=-2.1, z=1),   # rotate around X/Y/Z
        up=dict(x=0, y=0, z=1.5)       # z-axis is vertical
    )
)

# fig = go.FigureWidget(fig)

# show the fig and attach listener
display(fig)

# Attach a listener that prints the camera dict when you rotate
from IPython.display import Javascript

display(Javascript("""
require(["base/js/namespace"], function(Jupyter) {
  Jupyter.notebook.kernel.execute("last_camera = None")  // initialize
  const figs = document.querySelectorAll('.js-plotly-plot');
  figs.forEach(fig => {
    fig.on('plotly_relayout', ev => {
      if(ev['scene.camera']){
        const cam = JSON.stringify(ev['scene.camera']);
        Jupyter.notebook.kernel.execute("last_camera = " + cam);
        console.log("Camera updated:", ev['scene.camera']);
      }
    });
  });
});
"""))

# fig.show()
# fig.write_image(f'./Figure3/pca_proj_3d.png)
# fig.write_image("pca_projection.png", width=4400, height=3400, scale=1)

In [ ]:
# for leg in [False,True]:
#     for col in ['learn','geno_smpl']:
#         cats = pd.Index(sorted(proj_df[col].unique()))
#         n = len(cats)
#         palette = sns.color_palette("tab20", n) if n <= 20 else sns.husl_palette(n)
#         color_map = dict(zip(cats, palette))
#         point_colors = [mcolors.to_hex(color_map[g]) for g in proj_df[col]]

#         # --- Helper to save a 2D scatter as SVG ---
#         def save_pc2d(proj, pca_obj, xpc="PC1", ypc="PC2", fname="plot.svg"):
#             fig, ax = plt.subplots(figsize=(7.5, 6))
#             ax.scatter(proj[xpc], proj[ypc], s=50, alpha=0.9, c=point_colors)

#             # Axis labels with variance explained
#             pc_names = ["PC1","PC2","PC3"]
#             exp = {pc_names[i]: pca_obj.explained_variance_ratio_[i] for i in range(3)}
#             ax.set_xlabel(f"{xpc} ({exp[xpc]:.1%} var)")
#             ax.set_ylabel(f"{ypc} ({exp[ypc]:.1%} var)")
#             ax.set_title(f"PCA: {ypc} vs {xpc}")

#             # Legend (handles many genotypes)
#             handles = [plt.Line2D([0],[0], marker='o', linestyle='', markersize=7,
#                                 markerfacecolor=mcolors.to_hex(color_map[g])) for g in cats]
#             ncol = 1 if n < 12 else 2 if n <= 24 else 3
#             if leg:
#                 ax.legend(handles, list(cats), title=col,
#                         bbox_to_anchor=(1.02, 1), loc="upper left", ncol=ncol, fontsize=8)

#             ax.grid(True, alpha=0.2)
#             plt.tight_layout()
#             fig.savefig(fname, format="svg", dpi=300, bbox_inches="tight")
#             plt.close(fig)
#             print(f"Saved {fname}")

#         # --- Make the two figures ---
#         save_pc2d(proj_df, pca, xpc="PC1", ypc="PC2", fname=f"./Figure2/pca_PC2_vs_PC1_smpl_{col}_legend_{leg}.png")
#         save_pc2d(proj_df, pca, xpc="PC1", ypc="PC3", fname=f"./Figure2/pca_PC3_vs_PC1_smpl_{col}_legend_{leg}.png")


In [ ]:
loadings = pd.DataFrame(
    pca.components_,
    index=[f"PC{i+1}" for i in range(pca.n_components_)],
    columns=X.columns
)

# Optional: variance-scaled loadings (correlations) sometimes used for “importance”:
# These are eigenvectors scaled by sqrt(eigenvalues)
var_scaled_loadings = pd.DataFrame(
    (pca.components_.T * np.sqrt(pca.explained_variance_)).T,
    index=[f"PC{i+1}" for i in range(pca.n_components_)],
    columns=X.columns
)

# Canonical feature order: sorted by |PC1 loading|
pc1_order = loadings.loc["PC1"].abs().sort_values(ascending=False).index

for pc in ["PC1", "PC2", "PC3"]:
    pc_l = loadings.loc[pc].reindex(pc1_order)

    fig = plt.figure(figsize=(10, 5))
    pc_l.plot(kind="bar")
    plt.title(f"{pc} Loadings (ordered by |PC1 loading|)")
    plt.ylabel("Loading")
    plt.xlabel("Features (PC1-sorted)")
    plt.tight_layout()

    fig.savefig(f'./{control_dir}/{pc}_simplified.svg')

# # --- 4) Plot PC1 loadings sorted by absolute magnitude ---
# for pc in ["PC1",'PC2','PC3']:
#     pc_l = loadings.loc[pc].sort_values(key=np.abs, ascending=False)

#     fig = plt.figure(figsize=(10, 5))
#     pc_l.plot(kind="bar")
#     plt.title(f"{pc} Loadings (feature weights)")
#     plt.ylabel("Loading")
#     plt.xlabel("Features (sorted by |loading|)")
#     plt.tight_layout()

#     # If you want the top-N features programmatically:
#     topN = 15
#     # pc1_top = pc_l.head(topN)
#     # print(pc1_top)
#     fig.savefig(f'./{control_dir}/{pc}_simplified.svg')

In [ ]:
pc_l
np.linalg.norm(pc_l)


# LDA on the above factors

In [ ]:
# lda_df = op_df.loc[op_df['learn_class']<2]
# lda_df['learn_class']

lda_df = op_df.copy()
lda_df['control'] = lda_df['control'].map({True:0,False:1})
# lda_df['control'].astype(int).values

In [ ]:
labels = ['geno_smpl','learn_class','learn','note_text']

factors = [
    'outcome_fractions_no_as_no_mv',        # pc1, 3
    #'outcome_fractions_no_as_mv',          # pc3, 2
    'outcome_fractions_as_off',            # pc2, 1
    #'outcome_fractions_as_off_late',
    #'outcome_fractions_timeout',           # pc2, 4
    #'hi_lo_shift',
    'lo_state_on_target',                   # pc1, 2
    'hi_state_on_target',                   # pc1, 1
    'lo_target_off_state',                  # pc3, 1
    'hi_target_off_state',                  # pc3, 4
    'success_rate',                         # pc2, 5
    'hard_success_rate',                    # pc1, 4
    'as_off_over_successes',                # pc3, 4
    'as_off_over_hard_successes',           # pc1, 5
    'rms_velocity_bar',                     # pc2, 3
    #'holding_cost_bar',
    'probe_positive_effort_bar',            # pc2, 2, pc3, 5
]

X = lda_df[factors].values
y = lda_df['control'].astype(int).values

X_ = op_df[factors].values
y_ = op_df['control'].astype(int).values

scaler = StandardScaler()
x_scaled = scaler.fit_transform(X,)
x_scaled_ = scaler.fit_transform(X_,)

pca = PCA(n_components=10,
          copy=True, 
          whiten=False, 
          svd_solver='full', 
          tol=0.0, 
          iterated_power='auto')
Z = pca.fit_transform(x_scaled)
Z_ = pca.fit_transform(x_scaled_)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, roc_auc_score

# --- 1) Fit LDA in the PCA space
lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
lda.fit(Z, y)

# Binary LDA => one row in coef_
w_pc = lda.coef_.ravel()                 # direction in PC space (not unit)
b_pc = lda.intercept_.item()             # bias in PC space
w_pc_unit = w_pc / np.linalg.norm(w_pc)  # unit vector (direction only)

# Discriminant scores and predictions using the true linear form
scores = Z @ w_pc + b_pc                 # same as lda.decision_function(Z)
y_pred = (scores >= 0).astype(int)

print("Acc:", accuracy_score(y, y_pred))
print("ROC-AUC:", roc_auc_score(y, scores))

# --- 2) (Optional) Project onto the *unit* axis if you prefer a centered 1D view
scores_unit = Z @ w_pc_unit
# Class means along the unit axis (for plotting/intuition)
mu0 = scores_unit[y==0].mean()
mu1 = scores_unit[y==1].mean()

# --- 3) Map direction back to original features
# PCA components_: shape (n_pcs, n_features); rows are PC loadings
V = pca.components_                      # (k, p) here k=3, p=len(factors)
# Direction in original (scaled) feature space:
w_orig_scaled = V.T @ w_pc_unit          # (p,)

# If you want *raw units*, undo the StandardScaler scaling
w_orig_raw = w_orig_scaled / scaler.scale_
w_orig_raw /= np.linalg.norm(w_orig_raw) # unit vector in raw units

# Nice, readable series
w_scaled_s = pd.Series(w_orig_scaled, index=factors).sort_values(ascending=False)
w_raw_s    = pd.Series(w_orig_raw,    index=factors).sort_values(ascending=False)

print("\nTop drivers (scaled feature space):")
display(w_scaled_s.head(10))
print("\nTop drivers (raw units):")
display(w_raw_s.head(10))

# --- 4) Handy objects you might reuse
lda_results = {
    "w_pc": w_pc,               # discriminant direction (PC space)
    "b_pc": b_pc,               # intercept
    "w_pc_unit": w_pc_unit,     # unit direction (PC space)
    "scores": scores,           # linear discriminant scores (w·z + b)
    "scores_unit": scores_unit, # projection on unit axis (no intercept)
    "y_pred": y_pred,
    "w_orig_scaled": w_orig_scaled,  # feature contributions (scaled space)
    "w_orig_raw": w_orig_raw,        # feature contributions (raw units)
    'scaler':       x_scaled_,  # for scaling mutants
    'pcamodel':     pca,        # pca model for transforming factors
    'lda':          lda         # lda model fit to pcs.
}


In [ ]:
ftrfig = Figure(figsize=(4,2))
# plt.figure(figsize=(5,5))
ftrax = ftrfig.add_subplot(1, 1, 1)

ftrax = w_raw_s.plot(kind="barh", ax=ftrax)
ftrax.set_title("Feature contributions toward the '1' class (raw units)")
ftrax.set_xlabel("Loading along LDA direction")
ftrax.invert_yaxis()
ftrfig.tight_layout()
display(ftrfig)

In [ ]:
ftrfig = Figure(figsize=(4,2))
# plt.figure(figsize=(5,5))
ftrax = ftrfig.add_subplot(1, 1, 1)

ftrax = w_scaled_s.plot(kind="barh", ax=ftrax)
ftrax.set_title("Feature contributions toward the '1' class (scaled units)")
ftrax.set_xlabel("Loading along LDA direction")
ftrax.invert_yaxis()
ftrfig.tight_layout()
display(ftrfig)
ftrfig.savefig(fname=f'./{control_dir}/LDA_w_scaled.svg',format='svg')

In [ ]:
np.linalg.norm(w_scaled_s)


In [ ]:
w_pc_unit

In [ ]:
tags = sinq.all_tags()
print(tags)

In [ ]:
expts = [
    '1s_stim',
    'replay',
    'norpa',
    ['norpa','no_antenna'],
    'easy_targets',
    'no_antenna',
    'vnc_cut'
]

In [ ]:
for pc2idx in [2]: # [1,2]:
    pc12 = Z_[:,[0,pc2idx]]
    op_df["PC1"] = Z_[:,0]
    op_df["PC2"] = Z_[:,pc2idx]

    origin = pc12.mean(axis=0)
    # w2 = w_pc_unit[:2]
    w2 = w_pc[[0,pc2idx]]

    scale = 2.0 * pc12.std(0).mean()

    # decision boundary: {z : w·z + b = 0} → line with normal w2
    # pick a point x0 on the boundary in 2D by solving w2·x0 + (w_pc[2:]*mean(others) + b) ≈ 0
    # Since we're only plotting (PC1, PC2), treat other PCs at their means (≈0 after PCA).
    b2 = b_pc  # others ~ 0 in Z, so intercept stays b
    # Point on line: x0 = -b2 * w2 / ||w2||^2
    x0 = -b2 * w2 / (np.linalg.norm(w2)**2 + 1e-12)
    # Direction along the line is perpendicular to w2:
    v = np.array([-w2[1], w2[0]])
    t = np.linspace(-2, 2, 200)
    line = x0[None,:] + t[:,None]*v[None,:]

    op_df["_tags_str"] = op_df["tags"].apply(
        lambda t: ", ".join([t]) if t else ""
    )

    op_df["_hover"] = (
        "index: " + op_df.index.astype(str)
        + "<br>tags: " + op_df["_tags_str"]
    )

    import plotly.graph_objects as go

    color_map = {
        0: "#000000",   # blue
        1: "#989898",   # orange
        2: "#484848",   # orange
    }

    color_map = {
        False: "#000000",   # blue
        True: "#989898",   # orange
    }

    for expt in expts:
        
        fig = go.Figure()

        fig.add_trace(
            go.Scatter(
                x=op_df["PC1"],
                y=op_df["PC2"],
                mode="markers",
                marker=dict(
                    size=8,
                    color=op_df["control"].map(color_map),
                    opacity=0.7,
                ),
                hovertext=op_df["_hover"],
                hoverinfo="text",
                name="All points",
            )
        )

        labels = sinq.rows_with_tags(expt)
        print(f'Plotting {len(labels)} flies in {expt} experiments')
        sel_df = op_df.loc[labels]

        fig.add_trace(
            go.Scatter(
                x=sel_df["PC1"],
                y=sel_df["PC2"],
                mode="markers",
                marker=dict(
                    size=9,
                    color="#FFB000",
                    opacity=0.95,
                ),
                hovertext=sel_df["_hover"],
                hoverinfo="text",
                name=f"tagged: {expt}",
            )
        )

        fig.add_trace(
            go.Scatter(
                x=[origin[0], origin[0] + scale * w2[0]],
                y=[origin[1], origin[1] + scale * w2[1]],
                mode="lines+markers",
                line=dict(width=3),
                name="LDA direction",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=line[:, 0],
                y=line[:, 1],
                mode="lines",
                line=dict(dash="dash", width=2),
                name="decision boundary",
            )
        )
        fig.update_layout(
            title="LDA in PCA space",
            xaxis_title="PC1",
            yaxis_title="PC"+str(pc2idx+1),
            width=800,
            height=800,
            hovermode="closest",
        )

        fig.update_yaxes(scaleanchor="x", scaleratio=1)

        if isinstance(expt, str):
            es = expt
        elif isinstance(expt,list):
            es = '_'.join(expt)
        for fmt in ['html','svg','png']:
            fmt_dir = f"./{control_dir}/expts_in_LDA_space/{fmt}"
            os.makedirs(fmt_dir,exist_ok=True)
            fname = f"{fmt_dir}/LDA_in_PC1_PC{pc2idx+1}_{es}.{fmt}"
            if fmt=='html': 
                fig.write_html(fname)
            else:
                fig.write_image(fname)

In [ ]:
display(fig)

In [ ]:
np.linalg.norm(w_pc_unit)

In [ ]:
sinq.rows_with_tags('replay')

In [ ]:
# from sklearn.metrics import roc_curve
# fpr, tpr, thr = roc_curve(y, scores)
# j = tpr - fpr
# threshold = thr[np.argmax(j)]  # Youden's J

# # Predicted class by the 1D rule
# y_pred = (scores >= threshold).astype(int)
# y_pred

In [ ]:
# scores = lda.decision_function(Z)  # = Z @ w_pc + b, 1D array
scores = lda.decision_function(Z_)  # = Z @ w_pc + b, 1D array
op_df['lda_scores'] = scores
thr = 0.0

# fig, ax = plt.subplots(figsize=(5,3.2))
hstfig = Figure(figsize=(5,3.2))
# plt.figure(figsize=(5,5))
hstrax = hstfig.add_subplot(1, 1, 1)

for cls in (0, 1):
    # hstrax.hist(scores[y==cls], bins=16,  alpha=0.55, label=f"class {cls}")
    _, bins,_ = hstrax.hist(scores[y_==cls], bins=16,  alpha=0.55, label=f"class {cls}")

hstrax.axvline(thr, linestyle="--", linewidth=2)
hstrax.set_xlabel("LDA score  (w·z + b)")
hstrax.set_ylabel("Density")
hstrax.legend(frameon=False)
hstrax.set_title("Projection onto LDA direction with decision threshold")
hstfig.tight_layout()
display(hstfig)
hstfig.savefig(f'./{control_dir}/flies_onto_LDA.svg')

In [ ]:
op_df.columns

In [ ]:
op_df.loc[op_df['lda_scores']>=0,['lda_scores','tags','learn','note_text']]

# Show a couple of examples across the border

In [ ]:
'251107_F2_C1'
'251103_F2_C1'

In [ ]:
op_df.loc[['251107_F2_C1','251103_F2_C1'],['lda_scores','tags','learn','note_text']]

# Import mutant sinq and select other rows

In [ ]:
sinq_g = Sinq(sinqname='Figure3')
print(sinq_g.__repr__())

In [ ]:
g_idx = sinq_g.df.loc[~(sinq_g.df.index.isin(sinq.df.index))].index

In [ ]:
sinq_g = subset_sinq(source=sinq_g,rows=g_idx,name='Figure6_mutant_pca')

In [ ]:
sinq_g.df

In [ ]:
g_df = sinq_g.display_df(show_tags=True)
g_df['learn'] = sinq.has_tag('learn')
g_df['control'] = sinq.has_tag('control')
g_df[['genotype','tags','learn','control']]

In [ ]:
# print([i for i in op_df['genotype'].unique()])
list(g_df['genotype'].unique())

In [ ]:
gm_geno_map = {
 'iav_Kir2_1':                      'w;iav>kir2.1',
 '31H05_pJFRC7':                    'w',
 'iav-Gal4_+;+;UAS-Kir2_1':         '+;iav>kir2.1',
 '+;iav-Gal4_UAS-Kir2_1':           '+;iav>kir2.1',
 '+;31H05-Gal4_pJFRC7':             '+',
 '+;TH-Gal4_UAS-Kir2_1':            '+;TH>kir2.1',
 '+;TH-Gal4_pJFRC7':                '+',
 '+;UAS-TeTx;58E02-GAL4':           '+;UAS-TeTx;58E02-GAL4',
 }

g_df['geno_smpl'] = g_df['genotype'].map(gm_geno_map)
g_df.loc[g_df['geno_smpl'].isna()]

g_list = list(g_df['geno_smpl'].unique())
g_list

## Additional factors

In [ ]:
g_df['learn'] = 2  # Control
g_df['control'] = False
g_df['learn_class'] = 3 # testing if they are learners or not

In [ ]:
g_df['success_rate'] =         g_df['successes'] / g_df['num_trials']
g_df['hard_success_rate'] =    g_df['hard_successes'] / g_df['num_trials']

g_df['as_off_over_successes'] =        np.log10(g_df['outcome_fractions_as_off'] * g_df['num_trials'] + 1) / (g_df['successes'] + 1)
g_df['as_off_over_hard_successes'] =   np.log10(g_df['outcome_fractions_as_off'] * g_df['num_trials'] + 1) / (g_df['hard_successes'] + 1)

g_df['rms_velocity_bar'] = g_df['rms_velocity'] / g_df['duration']
g_df['holding_cost_bar'] = g_df['holding_cost'] / g_df['duration']
g_df['probe_positive_effort_bar'] = g_df['probe_positive_effort'] / g_df['duration']

# Project mutants into same space with LDA

In [ ]:
# lda_df = op_df.loc[op_df['learn_class']<2]
# lda_df['learn_class']

lda_df = g_df.copy()
# lda_df['control'] = lda_df['control'].map({True:0,False:1})
# lda_df['control'].astype(int).values

In [ ]:
labels = ['geno_smpl','learn_class','learn','note_text']

factors = [
    'outcome_fractions_no_as_no_mv',        # pc1, 3
    #'outcome_fractions_no_as_mv',          # pc3, 2
    'outcome_fractions_as_off',            # pc2, 1
    #'outcome_fractions_as_off_late',
    #'outcome_fractions_timeout',           # pc2, 4
    #'hi_lo_shift',
    'lo_state_on_target',                   # pc1, 2
    'hi_state_on_target',                   # pc1, 1
    'lo_target_off_state',                  # pc3, 1
    'hi_target_off_state',                  # pc3, 4
    'success_rate',                         # pc2, 5
    'hard_success_rate',                    # pc1, 4
    'as_off_over_successes',                # pc3, 4
    'as_off_over_hard_successes',           # pc1, 5
    'rms_velocity_bar',                     # pc2, 3
    #'holding_cost_bar',
    'probe_positive_effort_bar',            # pc2, 2, pc3, 5
]

# Put the new data in to the same coordinates as before
X_g = g_df[factors].values
y_g = g_df['control'].astype(int).values

x_scaled_g = scaler.transform(X_g)

Z_g = pca.transform(x_scaled_g)

In [ ]:
### ------ Don't need most of this because it's been done ------

# --- 1) Fit LDA in the PCA space
# lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
# lda.fit(Z, y)

# Binary LDA => one row in coef_
# w_pc = lda.coef_.ravel()                 # direction in PC space (not unit)
# b_pc = lda.intercept_.item()             # bias in PC space
# w_pc_unit = w_pc / np.linalg.norm(w_pc)  # unit vector (direction only)

# Discriminant scores and predictions using the true linear form
scores_g = Z_g @ w_pc + b_pc                 # same as lda.decision_function(Z)
y_pred_g = (scores >= 0).astype(int)

# print("Acc:", accuracy_score(y, y_pred))
# print("ROC-AUC:", roc_auc_score(y, scores))

# --- 2) (Optional) Project onto the *unit* axis if you prefer a centered 1D view
scores_unit_g = Z_g @ w_pc_unit
# Class means along the unit axis (for plotting/intuition)
# mu0 = scores_unit[y==0].mean()
# mu1 = scores_unit[y==1].mean()

# --- 3) Map direction back to original features
# PCA components_: shape (n_pcs, n_features); rows are PC loadings
# V = pca.components_                      # (k, p) here k=3, p=len(factors)
# Direction in original (scaled) feature space:
# w_orig_scaled = V.T @ w_pc_unit          # (p,)

# If you want *raw units*, undo the StandardScaler scaling
# w_orig_raw = w_orig_scaled / scaler.scale_
# w_orig_raw /= np.linalg.norm(w_orig_raw) # unit vector in raw units

# Nice, readable series
# w_scaled_s = pd.Series(w_orig_scaled, index=factors).sort_values(ascending=False)
# w_raw_s    = pd.Series(w_orig_raw,    index=factors).sort_values(ascending=False)

# print("\nTop drivers (scaled feature space):")
# display(w_scaled_s.head(10))
# print("\nTop drivers (raw units):")
# display(w_raw_s.head(10))

# --- 4) Handy objects you might reuse
# lda_results = {
#     "w_pc": w_pc,               # discriminant direction (PC space)
#     "b_pc": b_pc,               # intercept
#     "w_pc_unit": w_pc_unit,     # unit direction (PC space)
#     "scores": scores,           # linear discriminant scores (w·z + b)
#     "scores_unit": scores_unit, # projection on unit axis (no intercept)
#     "y_pred": y_pred,
#     "w_orig_scaled": w_orig_scaled,  # feature contributions (scaled space)
#     "w_orig_raw": w_orig_raw,        # feature contributions (raw units)
#     'scaler':       x_scaled_,  # for scaling mutants
#     'pcamodel':     pca,        # pca model for transforming factors
#     'lda':          lda         # lda model fit to pcs.
# }


In [ ]:
g_df["_tags_str"].head()

In [ ]:
def _get_gstr(g):
    g = g.replace('.', '_')
    g = g.replace('>', '_')
    g = g.replace('/', '_')
    g = g.replace('[', '')
    g = g.replace(']', '')
    g = g.replace('*', '')
    return g

for pc2idx in [2]: # [1,2]:
    pc12 = Z_[:,[0,pc2idx]]
    op_df["PC1"] = Z_[:,0]
    op_df["PC2"] = Z_[:,pc2idx]

    pc12_g = Z_g[:,[0,pc2idx]]
    g_df["PC1"] = Z_g[:,0]
    g_df["PC2"] = Z_g[:,pc2idx]

    origin = pc12.mean(axis=0)
    # w2 = w_pc_unit[:2]
    w2 = w_pc[[0,pc2idx]]

    scale = 2.0 * pc12.std(0).mean()

    # decision boundary: {z : w·z + b = 0} → line with normal w2
    # pick a point x0 on the boundary in 2D by solving w2·x0 + (w_pc[2:]*mean(others) + b) ≈ 0
    # Since we're only plotting (PC1, PC2), treat other PCs at their means (≈0 after PCA).
    b2 = b_pc  # others ~ 0 in Z, so intercept stays b
    # Point on line: x0 = -b2 * w2 / ||w2||^2
    x0 = -b2 * w2 / (np.linalg.norm(w2)**2 + 1e-12)
    # Direction along the line is perpendicular to w2:
    v = np.array([-w2[1], w2[0]])
    t = np.linspace(-2, 2, 200)
    line = x0[None,:] + t[:,None]*v[None,:]

    g_df["_tags_str"] = g_df["geno_smpl"].apply(
        lambda t: ", ".join([t]) if t else ""
    )

    g_df["_hover"] = (
        "index: " + g_df.index.astype(str)
        + "<br>tags: " + g_df["_tags_str"]
    )

    color_map = {
        0: "#000000",   # blue
        1: "#989898",   # orange
        2: "#484848",   # orange
    }

    color_map = {
        False: "#000000",   # blue
        True: "#989898",   # orange
    }

    for g_smpl in g_list:
        
        fig = go.Figure()

        # controls
        fig.add_trace(
            go.Scatter(
                x=op_df["PC1"],
                y=op_df["PC2"],
                mode="markers",
                marker=dict(
                    size=8,
                    color=op_df["control"].map(color_map),
                    opacity=0.7,
                ),
                hovertext=op_df["_hover"],
                hoverinfo="text",
                name="All points",
            )
        )
        
        # all mutants
        fig.add_trace(
            go.Scatter(
                x=g_df["PC1"],
                y=g_df["PC2"],
                mode="markers",
                marker=dict(
                    size=8,
                    color="#00C8C8",
                    opacity=0.7,
                ),
                hovertext=g_df["_hover"],
                hoverinfo="text",
                name="All points",
            )
        )

        # labels = sinq.rows_with_tags(expt)
        g_idx = g_df[g_df['geno_smpl']==g_smpl].index
        print(f'Plotting {len(g_idx)} flies in {g_smpl} experiments')
        sel_df = g_df.loc[g_idx]

        fig.add_trace(
            go.Scatter(
                x=sel_df["PC1"],
                y=sel_df["PC2"],
                mode="markers",
                marker=dict(
                    size=9,
                    color="#FFB000",
                    opacity=0.95,
                ),
                hovertext=sel_df["_hover"],
                hoverinfo="text",
                name=f"tagged: {g_smpl}",
            )
        )

        fig.add_trace(
            go.Scatter(
                x=[origin[0], origin[0] + scale * w2[0]],
                y=[origin[1], origin[1] + scale * w2[1]],
                mode="lines+markers",
                line=dict(width=3),
                name="LDA direction",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=line[:, 0],
                y=line[:, 1],
                mode="lines",
                line=dict(dash="dash", width=2),
                name="decision boundary",
            )
        )
        fig.update_layout(
            title="LDA in PCA space",
            xaxis_title="PC1",
            yaxis_title="PC"+str(pc2idx+1),
            width=800,
            height=800,
            hovermode="closest",
        )

        fig.update_yaxes(scaleanchor="x", scaleratio=1)
        fig.update_xaxes(range=[-6, 5])
        fig.update_yaxes(range=[-3, 3])

        if isinstance(g_smpl, str):
            gs = _get_gstr(g_smpl)
        elif isinstance(g_smpl,list):
            gs = _get_gstr('_'.join(g_smpl))

        for fmt in ['html','svg','png']:
            fmt_dir = f"./{mutants_dir}/expts_in_LDA_space/{fmt}"
            os.makedirs(fmt_dir,exist_ok=True)
            fname = f"{fmt_dir}/LDA_in_PC1_PC{pc2idx+1}_{gs}.{fmt}"
            if fmt=='html': 
                fig.write_html(fname)
            else:
                fig.write_image(fname)

In [ ]:
display(fig)

In [ ]:
w_pc_unit

In [ ]:
np.linalg.norm(w_pc_unit)

In [ ]:
# from sklearn.metrics import roc_curve
# fpr, tpr, thr = roc_curve(y, scores)
# j = tpr - fpr
# threshold = thr[np.argmax(j)]  # Youden's J

# # Predicted class by the 1D rule
# y_pred = (scores >= threshold).astype(int)
# y_pred

In [ ]:
# scores = lda.decision_function(Z)  # = Z @ w_pc + b, 1D array
scores = lda.decision_function(Z_g)  # = Z @ w_pc + b, 1D array
g_df['lda_scores'] = scores
thr = 0.0

# fig, ax = plt.subplots(figsize=(5,3.2))
hstfig = Figure(figsize=(5,3.2))
# plt.figure(figsize=(5,5))
hstrax = hstfig.add_subplot(1, 1, 1)

for cls in (0, 1):
    # hstrax.hist(scores[y==cls], bins=16,  alpha=0.55, label=f"class {cls}")
    hstrax.hist(scores, bins=bins,  alpha=0.55, label=f"class {cls}")

hstrax.axvline(thr, linestyle="--", linewidth=2)
hstrax.set_xlabel("LDA score  (w·z + b)")
hstrax.set_ylabel("Density")
hstrax.legend(frameon=False)
hstrax.set_title("Projection onto LDA direction with decision threshold")
hstfig.tight_layout()
display(hstfig)
hstfig.savefig(f'./{mutants_dir}/flies_onto_LDA.svg')

In [ ]:
g_df.loc[g_df['lda_scores']>=0,['lda_scores','tags','learn','note_text']]

# Statistical testing:

## Shuffle the target positions in the data
Redo all the classifications, and plot in the same space!

## Drop 10% of the learning and control flies
Compare the directions in PC space of the discriminant

## Compare to including the genetic mutants

## Just look at the first few blocks of each fly

## Just switch the labels on learners vs. non-learners and compare discriminants (cosine similarity)

# Extras

# Compare factors, learners vs control vs. mutants

In [ ]:
learn_df = op_df.copy()
learn_df['is_mutant'] = False

mutant_df = g_df.copy()
mutant_df['is_mutant'] = True

all_df = pd.concat([learn_df,mutant_df])

def assign_condition(df):
    df = df.copy()

    # example assumptions – adapt to your actual column names
    # df["is_mutant"] : bool
    # df["lda_score"] : float
    # lda_score > 0 => learner
    is_learner = df["lda_scores"] > 0

    df["condition"] = np.select(
        condlist=[
            (~df["is_mutant"]) & (~is_learner),
            (~df["is_mutant"]) & (is_learner),
            (df["is_mutant"]) & (~is_learner),
            (df["is_mutant"]) & (is_learner),
        ],
        choicelist=[
            "Control",
            "Learner (WT)",
            "Non-learning mutant",
            "Learning mutant",
        ],
        default="Unknown"
    )

    df["condition"] = pd.Categorical(
        df["condition"],
        categories=[
            "Control",
            "Learner (WT)",
            "Non-learning mutant",
            "Learning mutant",
        ],
        ordered=True
    )

    return df

all_df = assign_condition(all_df)
all_df[['condition','geno_smpl']]

order = [
    "Control",
    "Learner (WT)",
    "Non-learning mutant",
    "Learning mutant",
]

colors = {
    "Control": "#4C72B0",
    "Learner (WT)": "#55A868",
    "Non-learning mutant": "#C44E52",
    "Learning mutant": "#DD8452",
}

In [ ]:
def prep_factor_df(df, factor_col, condition_col):
    dfx = df[[factor_col, condition_col, "tags"]].copy()
    dfx = dfx.dropna(subset=[factor_col, condition_col])
    dfx["_index"] = dfx.index

    def format_tags(tags):
        return ", ".join(tags) if isinstance(tags, (list, tuple)) else ""

    dfx["_tags_str"] = dfx["tags"].apply(format_tags)
    return dfx


def plot_factor_box(
    df,
    factor_col,
    condition_col="condition",
    order=None,
    colors=None,
    fig_dir=factor_plots_dir,
    title=None,
):
    dfx = prep_factor_df(df, factor_col, condition_col)

    fig = go.Figure()

    for cond in order:
        sub = dfx[dfx[condition_col] == cond]

        fig.add_trace(
            go.Box(
                y=sub[factor_col],
                x=[cond] * len(sub),
                name=cond,
                marker_color=colors[cond],
                boxmean="sd",      # mean line + ±SD band
                boxpoints=False,   # NO individual points
            )
        )

    fig.update_layout(
        title=title or f"{factor_col} by condition",
        yaxis_title=factor_col,
        width=850,
        height=550,
        showlegend=False,
    )

    fig.write_image(f"{fig_dir}/box_{factor_col}.svg")
    os.makedirs(f"{fig_dir}/png/",exist_ok=True)
    fig.write_image(f"{fig_dir}/png/box_{factor_col}.png")
    return fig



def plot_factor_box_with_points(
    df,
    factor_col,
    condition_col="condition",
    order=None,
    colors=None,
    jitter=0.18,
    seed=0,
    fig_dir=factor_plots_dir,
    title=None,
):
    dfx = prep_factor_df(df, factor_col, condition_col)
    rng = np.random.RandomState(seed)

    fig = go.Figure()

    # --- boxplots ---
    for cond in order:
        sub = dfx[dfx[condition_col] == cond]

        fig.add_trace(
            go.Box(
                y=sub[factor_col],
                x=[cond] * len(sub),
                name=cond,
                marker_color=colors[cond],
                boxmean="sd",
                boxpoints=False,
                legendgroup=cond,
            )
        )

    # --- scatter points ---
    for i, cond in enumerate(order):
        sub = dfx[dfx[condition_col] == cond]
        xj = i + rng.uniform(-jitter, jitter, size=len(sub))

        fig.add_trace(
            go.Scatter(
                x=xj,
                y=sub[factor_col],
                mode="markers",
                marker=dict(
                    size=7,
                    color=colors[cond],
                    opacity=0.65,
                    line=dict(width=0),
                ),
                customdata=list(zip(sub["_index"], sub["_tags_str"])),
                hovertemplate=(
                    f"{cond}"
                    "<br>index: %{customdata[0]}"
                    f"<br>{factor_col}: %{{y:.4g}}"
                    "<br>tags: %{customdata[1]}"
                    "<extra></extra>"
                ),
                showlegend=False,
            )
        )

    fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(len(order))),
        ticktext=order,
    )

    fig.update_layout(
        title=title or f"{factor_col} by condition",
        yaxis_title=factor_col,
        width=900,
        height=600,
    )

    fig.write_image(f"{fig_dir}/boxpoints_{factor_col}.svg")
    os.makedirs(f"{fig_dir}/png/",exist_ok=True)
    os.makedirs(f"{fig_dir}/html/",exist_ok=True)
    fig.write_image(f"{fig_dir}/png/boxpoints_{factor_col}.png")
    fig.write_html(f"{fig_dir}/html/boxpoints_{factor_col}.html")
    return fig


In [ ]:
factors

In [ ]:
for factor in ["lo_state_on_target", "hi_state_on_target", "outcome_fractions_no_as_no_mv", 'as_off_over_hard_successes','rms_velocity_bar']:
    plot_factor_box(
        all_df, factor,
        order=order, colors=colors,
    )

    fig = plot_factor_box_with_points(
        all_df, factor,
        order=order, colors=colors,
    )

In [ ]:
display(fig)

In [ ]:
# all_df.loc['251103_F1_C1']
# g_df['as_off_over_successes'] =        np.log10(g_df['outcome_fractions_as_off'] * g_df['num_trials'] + 1) / (g_df['successes'] + 1)

all_df.loc['251103_F1_C1',['outcome_fractions_as_off','num_trials','successes']]

# Non parametric statistical tests

In [ ]:
from scipy.stats import kruskal



def kruskal_test(df, factor_col, condition_col="condition", order=None):
    groups = [
        df.loc[df[condition_col] == cond, factor_col].dropna()
        for cond in order
    ]

    H, p = kruskal(*groups)

    return {
        "test": "Kruskal–Wallis",
        "H": H,
        "p": p,
        "n_groups": len(groups),
    }


import itertools
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

def rank_biserial(u, n1, n2):
    return 1 - (2 * u) / (n1 * n2)

def pairwise_mwu(
    df,
    factor_col,
    condition_col="condition",
    order=None,
    correction="holm",
    alternative="two-sided",
):
    rows = []

    for a, b in itertools.combinations(order, 2):
        x = df.loc[df[condition_col] == a, factor_col].dropna()
        y = df.loc[df[condition_col] == b, factor_col].dropna()

        u, p = mannwhitneyu(x, y, alternative=alternative)

        rbc = rank_biserial(u, len(x), len(y))

        rows.append({
            "group1": a,
            "group2": b,
            "U": u,
            "p_uncorrected": p,
            "n1": len(x),
            "n2": len(y),
            "rank_biserial": rbc,
        })

    res = pd.DataFrame(rows)

    # multiple-comparison correction
    reject, p_corr, _, _ = multipletests(
        res["p_uncorrected"],
        method=correction
    )

    res["p_corrected"] = p_corr
    res["reject"] = reject

    return res.sort_values("p_corrected")



In [ ]:
factor = "hi_state_on_target"
kw = kruskal_test(all_df, factor, order=order)
print(kw)

pw = pairwise_mwu(
    all_df,
    factor,
    order=order,
)
pw